In [1]:
# Import necessary modules
import sys
from pathlib import Path
import cv2
from datetime import datetime

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from database import db
from student import student_manager
from attendance import attendance_manager
from recognition import recognizer
from embedding_generator import embedding_generator

2026-07-11 13:50:49,269 - database - INFO - Successfully connected to MongoDB database: attendance_system
2026-07-11 13:50:49,547 - numexpr.utils - INFO - NumExpr defaulting to 12 threads.
2026-07-11 13:50:54,320 - keras_facenet.embedding_model - INFO - Loading weights.
2026-07-11 13:50:54,322 - keras_facenet.utils - INFO - Looking for C:\Users\sanja/.keras-facenet\20180402-114759\20180402-114759-weights.h5
2026-07-11 13:50:59,831 - tensorflow - WARNING - From C:\Users\sanja\AppData\Roaming\Python\Python313\site-packages\keras\src\backend\tensorflow\core.py:233: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

2026-07-11 13:51:00,911 - embedding_generator - INFO - FaceNet model loaded successfully
2026-07-11 13:51:01,318 - recognition - WARNING - MediaPipe initialization failed: module 'mediapipe.tasks.python.vision.face_detector' has no attribute 'RunningMode'
2026-07-11 13:51:01,318 - recognition - INFO - Falling back to OpenCV Haar Cascade...
2026

In [2]:
# Check system readiness
print("System Readiness Check:")
print("-" * 40)

# Check registered students
students = student_manager.get_all_students()
print(f"✓ {len(students)} students registered")

# Check embeddings
embeddings = embedding_generator.load_all_embeddings()
print(f"✓ {len(embeddings)} embeddings loaded")

if len(embeddings) == 0:
    print("\n⚠ No embeddings found.")
    print("Please run 04_Generate_Embeddings.ipynb first.")

2026-07-11 13:51:01,371 - embedding_generator - INFO - Loaded 1 embeddings from database


System Readiness Check:
----------------------------------------
✓ 5 students registered
✓ 1 embeddings loaded


In [3]:
# Start attendance session

subject = input("Subject: ").strip() or "Machine Learning"
faculty = input("Faculty: ").strip() or "Dr. John Doe"
classroom = input("Classroom: ").strip() or "Room 201"

session_id = attendance_manager.start_attendance_session(
    subject,
    faculty,
    classroom
)

print(f"\n✓ Attendance session started: {session_id}")
print(f"Subject   : {subject}")
print(f"Faculty   : {faculty}")
print(f"Classroom : {classroom}")
print(f"Date      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\nStarting recognition...")
print("Press 'q' to finish attendance.")

Subject:  CD
Faculty:  Mr. PSK
Classroom:  SR 312


2026-07-11 13:51:22,810 - attendance - INFO - Attendance session started: ATT_20260711_135122_0dd561



✓ Attendance session started: ATT_20260711_135122_0dd561
Subject   : CD
Faculty   : Mr. PSK
Classroom : SR 312
Date      : 2026-07-11 13:51:22

Starting recognition...
Press 'q' to finish attendance.


In [4]:
# Run recognition with attendance

recognized_faces = []
capture_unknown = False

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("✗ Could not open webcam")

else:

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    try:

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            frame = cv2.flip(frame, 1)

            # Process frame
            display_frame, faces = recognizer.process_frame(
                frame,
                capture_unknown
            )

            # Store only unique recognized students
            for face in faces:

                if face["status"] == "recognized":

                    roll = face["roll_number"]

                    if not any(f["roll_number"] == roll for f in recognized_faces):

                        recognized_faces.append(face)

                        print(
                            f"✓ Recognized: "
                            f"{face['name']} "
                            f"({roll}) "
                            f"- {face['confidence']*100:.1f}%"
                        )

            # Display session id
            cv2.putText(
                display_frame,
                f"Session: {session_id}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1,
            )

            cv2.imshow("Real-Time Attendance", display_frame)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break

    finally:

        cap.release()
        cv2.destroyAllWindows()

2026-07-11 13:51:34,182 - tensorflow - WARNING - TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.


1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step
✓ Recognized: Rahul Kumar (22001) - 76.0%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 

In [5]:
# Mark attendance
print(f"\nFinishing attendance session...")
print(f"Recognized {len(recognized_faces)} unique students")

if recognized_faces:
    print("\nRecognized Students:")
    for face in recognized_faces:
        print(f"  {face.get('name', 'Unknown')} ({face.get('roll_number', 'N/A')}) - {face.get('confidence', 0)*100:.1f}%")

# Mark attendance - this now correctly handles the list
summary = attendance_manager.mark_attendance_for_session(session_id, recognized_faces)

print("\n" + "=" * 50)
print("ATTENDANCE SUMMARY")
print("=" * 50)
print(f"Total Students: {summary['total_students']}")
print(f"Present: {summary['present']}")
print(f"Absent: {summary['absent']}")
print(f"Attendance: {summary['attendance_percentage']:.1f}%")
print("=" * 50)
print(f"Session completed: {session_id}")

2026-07-11 13:51:48,762 - database - INFO - Attendance marked for student: 6a44ceca7eabfd5cf6b211f0
2026-07-11 13:51:48,769 - database - INFO - Attendance marked for student: 6a44ceca7eabfd5cf6b211f1
2026-07-11 13:51:48,774 - database - INFO - Attendance marked for student: 6a44ceca7eabfd5cf6b211f2
2026-07-11 13:51:48,778 - database - INFO - Attendance marked for student: 6a44ceca7eabfd5cf6b211f3
2026-07-11 13:51:48,784 - database - INFO - Attendance marked for student: 6a44ceca7eabfd5cf6b211f4
2026-07-11 13:51:48,791 - attendance - INFO - Attendance session ATT_20260711_135122_0dd561 completed. Present: 1, Absent: 4



Finishing attendance session...
Recognized 1 unique students

Recognized Students:
  Rahul Kumar (22001) - 76.0%

ATTENDANCE SUMMARY
Total Students: 5
Present: 1
Absent: 4
Attendance: 20.0%
Session completed: ATT_20260711_135122_0dd561


In [6]:
# View attendance records

records = attendance_manager.get_attendance_by_session(session_id)

print(f"\nAttendance Records ({len(records)} Records)")
print("-" * 60)

present_count = 0
absent_count = 0

for record in records[:10]:

    status = record["status"].upper()

    if status == "PRESENT":

        present_count += 1
        status_display = f"✓ {status}"

    else:

        absent_count += 1
        status_display = f"✗ {status}"

    print(
        f"{record['roll_number']} | "
        f"{record['name']:15} | "
        f"{status_display:10} | "
        f"{record.get('time_in','--')}"
    )

if len(records) > 10:
    print(f"...and {len(records)-10} more records")

print(f"\nSummary : {present_count} Present, {absent_count} Absent")


Attendance Records (5 Records)
------------------------------------------------------------
22001 | Rahul Kumar     | ✓ PRESENT  | 13:51:48
22002 | Priya Sharma    | ✗ ABSENT   | --
22003 | Sai Reddy       | ✗ ABSENT   | --
22004 | Krishna Kumar   | ✗ ABSENT   | --
22005 | Ravi Singh      | ✗ ABSENT   | --

Summary : 1 Present, 4 Absent


In [7]:
# Generate Excel report

from excel_report import excel_report

try:

    report_path = excel_report.generate_attendance_report(session_id)

    print(f"\n✓ Excel Report Generated")
    print(report_path)

except Exception as e:

    print(f"\n✗ Error : {e}")

2026-07-11 13:51:49,710 - excel_report - INFO - Attendance report generated: D:\attendance_system\Attendance\2026-07-11\Attendance_ATT_20260711_135122_0dd561_2026-07-11.xlsx



✓ Excel Report Generated
D:\attendance_system\Attendance\2026-07-11\Attendance_ATT_20260711_135122_0dd561_2026-07-11.xlsx


In [8]:
# Close database connection

db.close()

print("\n✓ Database connection closed")

2026-07-11 13:51:49,733 - database - INFO - MongoDB connection closed



✓ Database connection closed
